# Notebook 05 — Assembling the Telescope
## The full model, a training step, inference, and deployment · Paper §3–4

---
### Story so far → where we are now

We built every part in isolation:
- **NB 01:** the lens (warp maths Φ, Φ⁻¹, Jacobian).
- **NB 02:** mounting it on images (`FoveationWarpLayer`).
- **NB 03:** the calibration note for the detector (`HyperbolicEmbedding`).
- **NB 04:** detecting + un-warping + the training loss.

**NB 05 (here): we bolt all the parts together into one `TelescopeModel`** and run it
end-to-end — a forward pass, a real training step (with proper box-to-detective matching),
inference with timing, and notes on making it fast enough for a self-driving car.

### The assembled instrument, start to finish

```
A photo comes in
   │
   ├─ a thumbnail is glanced at to decide WHERE to aim the lens   → (o, R)   [NB 01]
   │                                              │
   ├─ the lens magnifies the distant region       (warp)          [NB 02]
   │                                              │
   ├─ the lens setting is written on a sticky note (embedding)    [NB 03]
   │                                              │
   ├─ a big pre-trained vision network looks at the warped photo  (backbone)
   │                                              │
   ├─ ~300 "detective" queries investigate it     (Transformer decoder)
   │                                              │
   ├─ each detective proposes a box in warped space (box head)    [NB 04]
   │                                              │
   └─ boxes are un-warped back to the real photo   (Φ⁻¹)          → final detections
```

Everything below is the same code that runs in `train.py` and `eval.py` — the notebook just
uses tiny dimensions so it runs on a laptop CPU in seconds. The real model swaps in the
SAM 3.1 backbone and a real Deformable DETR decoder (see the README), but the wiring is
identical.

In [ ]:
import sys, pathlib, time
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
import matplotlib.pyplot as plt
import numpy as np

torch.set_default_dtype(torch.float32)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {DEVICE}')

In [ ]:
from telescope import TelescopeModel, TelescopeLoss, denoise_boxes
from telescope.box import euclidean_to_riemannian_box
from telescope.matcher import HungarianMatcher, match_and_compute_loss
from telescope.eval import CocoEvaluator, DetectionResult

print('telescope package imported.')

---
## 1 · Instantiate and inspect `TelescopeModel`

In [ ]:
# Class convention: 6 foreground classes + 1 background (last index)
CLASS_NAMES = ['Car', 'Truck', 'Sign', 'Bike', 'Debris', 'Person', '__background__']
NUM_CLASSES = len(CLASS_NAMES)   # = 7

torch.manual_seed(0)
model = TelescopeModel(
    num_classes  = NUM_CLASSES,  # 6 foreground + background
    num_queries  = 20,           # paper: 300 — reduced for notebook speed
    query_dim    = 64,           # paper: 256 — reduced for notebook speed
    enc_out_dim  = 64,
)
model.to(DEVICE)

total  = sum(p.numel() for p in model.parameters())
frozen = sum(p.numel() for p in model.backbone.parameters())  # SAM3 frozen in real model
print(f'Total parameters     : {total:,}')
print(f'Backbone (stub) params: {frozen:,}  ← replaced by frozen SAM3 in production')
print()

# Named module summary
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f'  {name:<20}  {n:>8,} params')

---
## 2 · Forward pass

A single call through the whole pipeline, from raw image to Euclidean bounding boxes.

In [ ]:
# Simulate a batch of 2 images at 128×128 (paper uses 1024×1024)
dummy_images = torch.rand(2, 3, 128, 128).to(DEVICE)

model.eval()
with torch.no_grad():
    boxes_eu, class_logits, o, R = model(dummy_images)

print('Forward pass output shapes:')
print(f'  boxes_eu      : {boxes_eu.shape}      (B, Q, 4)  [cx, cy, w, h]')
print(f'  class_logits  : {class_logits.shape}  (B, Q, num_classes)')
print(f'  o (centres)   : {o.shape}           (B, 2)')
print(f'  R (radii)     : {R.shape}             (B,)')

# Predicted class and score for each query (exclude the background class = last)
probs = class_logits.softmax(-1)                     # (B, Q, C)
scores, pred_classes = probs[:, :, :-1].max(-1)      # ignore background class
print(f'\nBatch 0 top-3 detections by score:')
top3 = scores[0].topk(3)
for score, idx in zip(top3.values, top3.indices):
    cx, cy, w, h = boxes_eu[0, idx].tolist()
    print(f'  Q{idx:02d}  {CLASS_NAMES[pred_classes[0, idx]]:8}  '
          f'score={score:.3f}  box=[{cx:.2f}, {cy:.2f}, {w:.3f}, {h:.3f}]')

---
## 3 · One training step — and how to read the loss curve

Before the model can learn, we must answer a matchmaking question: we have ~300 detective
queries but maybe only 3 real objects in the image. *Which detective is responsible for which
object?* If we got this wrong, we'd punish a detective for missing an object it was never
assigned.

The **Hungarian matcher** solves this optimally. It looks at every possible detective-object
pairing, scores how good each would be (using the gIoU + L1 costs from NB 04), and picks the
one-to-one assignment with the lowest total cost — like assigning workers to jobs so the whole
team is most efficient. Unmatched detectives are simply told "you found nothing here", which
is correct most of the time.

Once matched, we compute the loss, back-propagate, and update the weights. One detail worth
noting: the big pre-trained backbone is **frozen** (we never change it) — only the small new
parts (lens-aimer, embedding, decoder, box head) actually learn. This is why training fits on
a modest GPU.

**How to read the loss curve below:** the x-axis is training steps, the y-axis is the loss
(lower = better predictions). Over these few demo steps on random data you won't see dramatic
learning — the point is simply that the curve is **finite, stable, and generally trending
down**, proving the whole chain (warp → backbone → decoder → box head → un-warp → loss →
gradients) is wired correctly and nothing is broken. Real training curves over thousands of
steps on real data are what produce the accuracy numbers in NB 06.

In [ ]:
matcher = HungarianMatcher(cost_cls=1.0, cost_l1=5.0, cost_giou=2.0)
# CLASS_NAMES and NUM_CLASSES were defined in the model cell above


def training_step(
    model:     TelescopeModel,
    optimizer: torch.optim.Optimizer,
    images:    Tensor,
    gt_boxes_list:  list,   # list of (M_b, 4) tensors per image
    gt_labels_list: list,   # list of (M_b,)   tensors per image
    max_grad_norm:  float = 0.1,
) -> dict:
    """One complete training iteration with Hungarian matching.

    Replaces the random-assignment placeholder used in earlier cells.
    """
    model.train()
    optimizer.zero_grad()

    boxes_eu, class_logits, o, R, boxes_ri = model(images, return_riemannian=True)

    losses = match_and_compute_loss(
        pred_boxes_ri   = boxes_ri,
        pred_logits     = class_logits,
        gt_boxes_list   = gt_boxes_list,
        gt_labels_list  = gt_labels_list,
        o               = o,
        R               = R,
        matcher         = matcher,
        num_classes     = NUM_CLASSES,
    )

    losses['loss_total'].backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
    optimizer.step()

    return {k: v.item() if hasattr(v, 'item') else v for k, v in losses.items()}


# ── Run a few steps to verify ─────────────────────────────────────────────────
torch.manual_seed(0)
model.train()
for p in model.backbone.parameters():
    p.requires_grad_(False)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4
)

B = 2
history = []

for step in range(6):
    images = torch.rand(B, 3, 64, 64).to(DEVICE)

    # Simulate a variable number of GT boxes per image (Hungarian handles any M)
    gt_boxes_list  = []
    gt_labels_list = []
    for _ in range(B):
        n  = torch.randint(2, 5, ()).item()
        gb = torch.rand(n, 4)
        gb[:, 2:] = gb[:, 2:].abs() * 0.2 + 0.02
        gb[:, :2] = gb[:, :2] * 0.8 - 0.4
        gt_boxes_list.append(gb)
        # foreground labels only: 0 .. NUM_CLASSES-2 (last index is background)
        gt_labels_list.append(torch.randint(0, NUM_CLASSES - 1, (n,)))

    result = training_step(model, optimizer, images, gt_boxes_list, gt_labels_list)
    history.append(result['loss_total'])
    print(f'  Step {step+1}  loss={result["loss_total"]:.4f}  '
          f'matched={result["n_matched"]}  '
          f'loss_l1={result["loss_l1"]:.3f}  loss_giou={result["loss_giou"]:.3f}')

plt.figure(figsize=(6, 3))
plt.plot(history, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('Training loss with Hungarian matching')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 4 · Inference and timing — how to read the latency breakdown

At inference (using the model, not training it) we run a photo through and keep only the
confident detections, discarding the detectives who found nothing.

A self-driving car needs this to happen *fast* — ideally 30+ times per second. So we time
each stage of the pipeline separately. The printout below lists milliseconds per stage:

- **Param estimation:** glancing at the thumbnail to aim the lens — should be tiny.
- **Warp:** magnifying the image. On full-resolution images this is the heaviest geometry
  step (it runs the inverse $\Phi^{-1}$ over every pixel) — we discuss speed tricks in §5.
- **Backbone encode / DETR decode:** the big neural-network stages — usually the bulk of the
  time on a real model.
- **Box head + un-warp:** turning detective findings into final boxes — tiny.

**How to read it:** add the stages up to get total latency; 1000 ÷ total = frames per second.
On this CPU stub the absolute numbers are meaningless, but the *relative* sizes tell you where
the time goes and therefore what to optimise first. The real-deployment numbers (≈30 ms total
on an A100 GPU) are in §5 and the README.

In [ ]:
@torch.no_grad()
def inference(
    model: TelescopeModel,
    image: Tensor,              # (1, 3, H, W)
    score_threshold: float = 0.3,
    class_names: list = None,
) -> dict:
    """Run inference and return filtered detections.

    Returns dict with:
        boxes    : (K, 4) Euclidean [cx, cy, w, h] for K confident detections
        scores   : (K,)
        labels   : (K,) integer class indices
        o, R     : foveation params used for this image
    """
    model.eval()
    boxes_eu, logits, o, R = model(image)

    probs   = logits[0].softmax(-1)          # (Q, C)
    scores, labels = probs[:, :-1].max(-1)   # ignore background

    mask   = scores > score_threshold
    return dict(
        boxes  = boxes_eu[0][mask],
        scores = scores[mask],
        labels = labels[mask],
        o=o[0], R=R[0],
    )


# ── Timing: measure latency of each stage ────────────────────────────────────
class Timer:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        self.t = time.perf_counter()
        return self
    def __exit__(self, *a):
        self.elapsed = (time.perf_counter() - self.t) * 1000
        print(f'  {self.name:<30}: {self.elapsed:6.1f} ms')

model.eval()
img = torch.rand(1, 3, 128, 128).to(DEVICE)

print('Stage latency breakdown (stub model, CPU):')
with torch.no_grad():
    # Warm-up
    _ = model(img)

    with Timer('Param estimation (Stage 1a)'):
        small = F.interpolate(img, (model.low_res_size, model.low_res_size), mode='bilinear', align_corners=False)
        feats = model.backbone(small)
        enc_vec = torch.cat([f.flatten(2).mean(-1) for f in feats], dim=-1)
        o, R = model.fov_estimator(enc_vec)

    with Timer('Warp (Stage 1b)'):
        warped = model.warp_layer(img, o, R)

    with Timer('Backbone encode (Stage 2a)'):
        features = model.backbone(warped)

    with Timer('DETR decode (Stage 2a)'):
        fov_params = torch.cat([o, R.unsqueeze(-1).expand(-1, 2)], dim=-1)
        embedding  = model.hyperbolic_emb(fov_params)
        queries    = model.object_queries.weight.unsqueeze(0)
        aug_q      = queries + embedding.unsqueeze(1)
        query_feats= model.detr(features, aug_q)

    with Timer('Box head + Phi^{-1} (Stage 2b)'):
        boxes_ri, logits = model.box_head(query_feats)
        boxes_eu_list = []
        for b_i in range(1):
            from telescope.box import riemannian_to_euclidean_box
            eu = riemannian_to_euclidean_box(boxes_ri[b_i], o[b_i], R[b_i])
            boxes_eu_list.append(eu)

print()
result = inference(model, img, score_threshold=0.1)
print(f'Detections above threshold: {len(result["boxes"])}')
print(f'Foveation centre: o={result["o"].tolist()}')
print(f'Foveation radius: R={result["R"].item():.3f}')

---
## 3.5 · Validation loop with COCO mAP

`CocoEvaluator` acumula predicciones y GT durante toda la época de validación
y luego calcula mAP con `pycocotools` (o un AP@0.5 simplificado si no está instalado).

```bash
pip install pycocotools   # para mAP completo COCO 0.5:0.95
```

In [ ]:
@torch.no_grad()
def validation_epoch(
    model:          TelescopeModel,
    val_loader,     # iterable of (images, targets) — see format below
    score_threshold: float = 0.05,
) -> dict:
    """Run one validation epoch and return COCO metrics.

    targets format (list of dicts per image):
        {
            'boxes':    Tensor (M, 4)  Euclidean [cx, cy, w, h]
            'labels':   Tensor (M,)    integer class indices
            'image_id': int
        }

    Returns dict: mAP, mAP_50, mAP_75, ...
    """
    model.eval()
    evaluator = CocoEvaluator(num_classes=NUM_CLASSES - 1,  # exclude background
                               class_names=CLASS_NAMES[:-1])

    for images, targets in val_loader:
        images = images.to(DEVICE)
        boxes_eu, logits, o, R = model(images)

        # Post-process: softmax scores, filter by threshold
        probs  = logits.softmax(-1)[:, :, :-1]   # exclude background
        scores, labels = probs.max(-1)

        for b, target in enumerate(targets):
            keep  = scores[b] > score_threshold
            result = DetectionResult(
                boxes    = boxes_eu[b][keep].cpu(),
                scores   = scores[b][keep].cpu(),
                labels   = labels[b][keep].cpu(),
                image_id = target['image_id'],
            )
            evaluator.update([result], [target])

    return evaluator.summarize()


# ── Demo: fake validation loader ─────────────────────────────────────────────
def make_fake_val_batch(B=2, n_boxes=3, H=64, W=64):
    images  = torch.rand(B, 3, H, W)
    targets = []
    for b in range(B):
        gb = torch.rand(n_boxes, 4)
        gb[:, 2:] = gb[:, 2:].abs() * 0.2 + 0.05
        gb[:, :2] = gb[:, :2] * 0.6 - 0.3
        targets.append({
            'boxes':    gb,
            'labels':   torch.randint(0, NUM_CLASSES - 1, (n_boxes,)),
            'image_id': b,
        })
    return images, targets

fake_val_loader = [make_fake_val_batch() for _ in range(4)]   # 4 batches

metrics = validation_epoch(model, fake_val_loader)
print('Validation metrics (random weights — numbers are meaningless):')
for k, v in metrics.items():
    print(f'  {k:<15}: {v:.4f}')

---
## 5 · Real-time deployment checklist

The following steps are required to go from this notebook to a production system.
They are documented here rather than implemented, because they depend on
licensed models and datasets not included in this repository.

### 5.1 Substituting the stub components with real weights

| Stub class | Real component | Where to get it |
|---|---|---|
| `SAM3EncoderStub` | SAM 2.1 ViT-H encoder (frozen) | `pip install sam2` from [facebookresearch/sam2](https://github.com/facebookresearch/sam2) |
| `DeformableDetrStub` | Deformable DETR decoder | [detrex](https://github.com/IDEA-Research/detrex) or [mmdetection](https://github.com/open-mmlab/mmdetection) |

See `README.md` for full download commands and weight paths.

### 5.2 Performance optimisation path

```
Baseline (this notebook, CPU)          ~200–500 ms / image
     ↓  GPU (A100)                     ~15–25 ms / image
     ↓  torch.compile()                ~12–18 ms
     ↓  ONNX export + TensorRT FP16    ~5–8 ms
     ↓  TensorRT INT8 calibration      ~3–5 ms
```

### 5.3 NR inverse at inference scale

At 1024×1024, the NR inverse runs on 1M+ grid points per image.  Options:
- **Pre-compute the grid** at the start of each batch (o, R are fixed → grid cached)
- **Reduce inference resolution** (paper: warp computed at training res, grid bilinearly upsampled)
- **CUDA kernel**: replace the Python NR loop with a fused CUDA kernel (reduces Python overhead)

The paper reports ~17ms for the full warp pipeline on an A100.

In [ ]:
# ── Tip 1: cache the source grid when (o, R) are fixed across a batch ────────
from telescope.warp import compute_source_grid
import torch.nn.functional as F

def warp_with_cached_grid(
    images: Tensor,     # (B, C, H, W)
    o: Tensor,          # (B, 2) — must be same for all images if you want to share grid
    R: Tensor,          # (B,)
    cached_grid=None,
):
    """Warp with optional grid caching.  If o, R don't change frame-to-frame
    (e.g., during evaluation), the source grid can be computed once."""
    B, C, H, W = images.shape
    if cached_grid is None:
        cached_grid = compute_source_grid(H, W, o, R)
    return F.grid_sample(images, cached_grid, mode='bilinear',
                          padding_mode='border', align_corners=True), cached_grid


# ── Tip 2: torch.compile() for 10–30% speedup on eager ops ──────────────────
# (requires PyTorch >= 2.0 and a GPU with CUDA)
def compile_model_for_inference(model: TelescopeModel) -> TelescopeModel:
    """Apply torch.compile to the expensive sub-modules."""
    if torch.cuda.is_available():
        model.backbone       = torch.compile(model.backbone)
        model.detr           = torch.compile(model.detr)
    else:
        print('torch.compile skipped: no CUDA device available')
    return model

print('Grid caching and torch.compile helpers defined.')


# ── Tip 3: verify output is identical with and without caching ───────────────
torch.manual_seed(0)
test_img = torch.rand(1, 3, 64, 64)
o_c = torch.tensor([[0.1, -0.05]])
R_c = torch.tensor([0.5])

warped_no_cache, grid = warp_with_cached_grid(test_img, o_c, R_c, cached_grid=None)
warped_cached,   _    = warp_with_cached_grid(test_img, o_c, R_c, cached_grid=grid)
assert torch.allclose(warped_no_cache, warped_cached)
print('PASS  cached grid produces identical output')

---
## 6 · ONNX export (production path)

Exporting to ONNX allows deployment on any hardware with an ONNX runtime
(TensorRT, OpenVINO, ONNX Runtime).  The NR inverse loop is the tricky part:
ONNX does not support dynamic control flow by default — we unroll the loop
to a fixed number of iterations.

```
★ Insight ─────────────────────────────────────
  • For ONNX export, replace HyperbolicInverseNR with a fixed-iteration
    version (no early stopping) so the computation graph is static.
    8 iterations is sufficient for error < 1e-6 (paper Appendix C).
  • The custom autograd Function is forward-only compatible with ONNX —
    the backward is irrelevant at inference time.
  • Use opset_version=17 or higher for full operator support.
─────────────────────────────────────────────────
```

In [ ]:
import io

class FixedIterNR(torch.nn.Module):
    """NR inverse with a FIXED number of iterations — ONNX-exportable.

    No early stopping → static compute graph → ONNX compatible.
    8 iterations is sufficient for error < 1e-6 for typical (alpha, R, p).
    """
    def __init__(self, alpha=2.0, p=2.0, eta=0.5, n_iters=8):
        super().__init__()
        self.alpha   = alpha
        self.p       = p
        self.eta     = eta
        self.n_iters = n_iters

    def forward(self, y: Tensor, o: Tensor, R: Tensor) -> Tensor:
        from telescope.geometry import hyperbolic_foveated_transform
        x = y.clone()
        for _ in range(self.n_iters):   # fixed iteration count — ONNX-safe
            x = x + self.eta * (y - hyperbolic_foveated_transform(x, o, R, self.alpha, self.p))
        return x


# Verify it gives similar output to the convergence-based NR
from telescope.geometry import hyperbolic_inverse

torch.manual_seed(0)
o_ex = torch.tensor([0.0, 0.0])
R_ex = torch.tensor(0.6)
y_ex = torch.rand(10, 2) * 0.6 - 0.3

fixed_nr = FixedIterNR(n_iters=8)
x_fixed   = fixed_nr(y_ex, o_ex, R_ex)
x_conv    = hyperbolic_inverse(y_ex, o_ex, R_ex)

max_diff = (x_fixed - x_conv).abs().max().item()
print(f'Fixed(8 iters) vs convergence NR: max diff = {max_diff:.2e}')
assert max_diff < 1e-5
print('PASS  fixed-iteration NR matches convergence NR')

# ONNX export of the fixed-iteration NR
try:
    buf = io.BytesIO()
    torch.onnx.export(
        fixed_nr,
        (y_ex, o_ex, R_ex),
        buf,
        input_names  = ['y', 'o', 'R'],
        output_names = ['x_star'],
        opset_version= 17,
        dynamo       = False,
    )
    size_kb = buf.tell() / 1024
    print(f'ONNX export succeeded  ({size_kb:.1f} KB)')
except Exception as e:
    print(f'ONNX export note: {e}')
    print('(Full model export requires resolving dynamic control flow — use fixed-iter version above)')

---
## Summary — the complete Telescope, and what's left

Every component from NBs 01–04 is now wired into one trainable, runnable model:

| Stage | Module | Built in | Real-world component |
|---|---|---|---|
| Aim the lens | `FoveationEstimator` | NB 01, 03 | small FFN on a thumbnail |
| Magnify the image | `FoveationWarpLayer` | NB 02 | `grid_sample` + Φ⁻¹ |
| Calibration note | `HyperbolicEmbedding` | NB 03 | added to decoder queries |
| Look at the image | `SAM3EncoderStub` | NB 05 | frozen SAM 3.1 backbone |
| Investigate objects | `DeformableDetrStub` | NB 05 | Deformable DETR decoder |
| Propose boxes | `RiemannianBoxHead` | NB 04 | MLP → warped-space boxes |
| Un-warp results | `riemannian_to_euclidean_box` | NB 01, 04 | Φ⁻¹ + Jacobian |
| Score & teach | `TelescopeLoss`, `HungarianMatcher` | NB 04, 05 | gIoU + L1 + matching |

**What this notebook did *not* do:** actually train to convergence. That needs real data and
hours of GPU time (run `train.py`, see the README). 

**Coming up in NB 06** — the experiment that justifies the whole project: after training, we
load the result, measure its accuracy at different distances, and run the model **with vs.
without** the foveation lens to answer the only question that matters — *does the telescope
actually help us see farther?*